Q9: Data Cleaning and Preparation

Analyzing customer churn in a telecommunications company.

Use Telecom_Customer_Churn.csv from the same folder. This is the complete original IBM Telco Customer Churn dataset with 7,043 rows and 21 columns.

Source: https://github.com/IBM/telco-customer-churn-on-icp4d/blob/master/data/Telco-Customer-Churn.csv

The original CSV is unchanged. Cleaned data is saved separately as Telecom_Customer_Churn_Cleaned.csv.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

data = pd.read_csv("Telecom_Customer_Churn.csv")

print(data.head())
data.info()
print("Shape:", data.shape)

   customerID  gender  SeniorCitizen  ... MonthlyCharges TotalCharges  Churn
0  7590-VHVEG  Female              0  ...          29.85        29.85     No
1  5575-GNVDE    Male              0  ...          56.95       1889.5     No
2  3668-QPYBK    Male              0  ...          53.85       108.15    Yes
3  7795-CFOCW    Male              0  ...          42.30      1840.75     No
4  9237-HQITU  Female              0  ...          70.70       151.65    Yes

[5 rows x 21 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-

In [2]:
# Standardize text and convert data types
for col in data.select_dtypes(include="object").columns:
    data[col] = data[col].str.strip().replace("", np.nan)

data["gender"] = data["gender"].str.lower()
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")

In [3]:
# Check and fill missing values
print("Missing Values:")
print(data.isnull().sum())

num_cols = data.select_dtypes(include=np.number).columns
data[num_cols] = data[num_cols].fillna(data[num_cols].median())

cat_cols = data.select_dtypes(include="object").columns
for col in cat_cols:
    data[col] = data[col].fillna(data[col].mode()[0])

data["SeniorCitizen"] = data["SeniorCitizen"].astype(int)

Missing Values:
customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64


In [4]:
# Remove duplicate records
data = data.drop_duplicates().copy()
print("After removing duplicates:", data.shape)

After removing duplicates: (7043, 21)


In [5]:
# Handle outliers using IQR
for col in ["tenure", "MonthlyCharges", "TotalCharges"]:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    data[col] = data[col].clip(lower, upper)

In [6]:
# Create new features
data["AvgMonthlyCharge"] = data["TotalCharges"] / (data["tenure"] + 1)
data["TenureGroup"] = pd.cut(
    data["tenure"],
    bins=[0, 12, 24, 48, 100],
    labels=["New", "Medium", "Long", "Very Long"],
    include_lowest=True
)

In [7]:
# Scale numerical columns
num_cols = data.select_dtypes(include=np.number).columns
scaler = StandardScaler()
data[num_cols] = scaler.fit_transform(data[num_cols])

print("Final Dataset:")
print(data.head())
print("\nFinal Shape:", data.shape)

Final Dataset:
   customerID  gender  SeniorCitizen  ... Churn AvgMonthlyCharge  TenureGroup
0  7590-VHVEG  female      -0.439916  ...    No        -0.757979          New
1  5575-GNVDE    male      -0.439916  ...    No        -0.117801         Long
2  3668-QPYBK    male      -0.439916  ...   Yes        -0.411755          New
3  7795-CFOCW    male      -0.439916  ...    No        -0.346750         Long
4  9237-HQITU  female      -0.439916  ...   Yes        -0.174110          New

[5 rows x 23 columns]

Final Shape: (7043, 23)


In [8]:
# Save the cleaned dataset
data.to_csv("Telecom_Customer_Churn_Cleaned.csv", index=False)
print("Saved Telecom_Customer_Churn_Cleaned.csv")

Saved Telecom_Customer_Churn_Cleaned.csv
